In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()


os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

Data Ingestion

In [ ]:
from langchain.document_loaders import TextLoader

In [ ]:
loader = TextLoader("../data/agenticAI.txt", encoding="utf8")
documents = loader.load()

In [ ]:
documents[0].page_content[:500]  # Print the first 500 characters of the first documen

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=20)
text_chunks=text_splitter.split_documents(documents)
text_chunks

In [ ]:
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS

embeddings=OpenAIEmbeddings()
vectorstore=FAISS.from_documents(text_chunks, embeddings)
vectorstore

In [ ]:
retriever=vectorstore.as_retriever()

In [ ]:
# Perform similarity search
query = "What is the Key Characteristics of Agentic AI?"
docs = vectorstore.similarity_search(query, k=4)

# Display the results
for i, doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(doc.page_content)
    print("-" * 50)

In [ ]:
from langchain.prompts import ChatPromptTemplate

template="""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""

In [ ]:
prompt=ChatPromptTemplate.from_template(template)
prompt

In [ ]:
from langchain.schema.output_parser import StrOutputParser
from langchain.chat_models import ChatOpenAI

output_parser=StrOutputParser()
llm_model=ChatOpenAI(model_name="gpt-4o-mini")

In [ ]:
from langchain.schema.runnable import RunnablePassthrough


rag_chain = (
    {"context": retriever,  "question": RunnablePassthrough()}
    | prompt
    | llm_model
    | output_parser
)
rag_chain.invoke("tell me about Agentic AI")